# Week 51, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w051d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import heapq

g = {"A": {"B": 4, "C": 1}, "C": {"B": 2, "D": 6}, "B": {"D": 1}, "D": {}}
dist, heap, order = {"A": 0}, [(0, "A")], []
while heap:
    d, u = heapq.heappop(heap)
    if u in order:
        continue
    order.append(u)
    for v, w in g[u].items():
        if d + w < dist.get(v, float("inf")):
            dist[v] = d + w
            heapq.heappush(heap, (d + w, v))
print(order, dist)

**Predict before you run:** what will this print?



In [ ]:
edges = [("S", "A", 3), ("A", "B", -2), ("S", "B", 2)]
dist = {"S": 0, "A": float("inf"), "B": float("inf")}
for _ in range(2):
    for u, v, w in edges:
        dist[v] = min(dist[v], dist[u] + w)
print(dist)

**Predict before you run:** what will this print?



In [ ]:
parent = {x: x for x in "abcd"}
def find(x):
    while parent[x] != x:
        x = parent[x]
    return x
kept = 0
for w, u, v in sorted([(3, "a", "b"), (1, "b", "c"), (2, "a", "c"), (4, "c", "d"), (5, "b", "d")]):
    if find(u) != find(v):
        parent[find(u)] = find(v)
        kept += w
print(kept)

**Predict before you run:** what will this print?



In [ ]:
goal = (3, 4)
h = lambda cell: abs(cell[0] - goal[0]) + abs(cell[1] - goal[1])
print(h((0, 0)), h((3, 0)), h(goal))

**Predict before you run:** what will this print?



In [ ]:
import math
rates = [2.0, 0.6, 0.9]
print(round(math.prod(rates), 3), round(sum(-math.log(r) for r in rates), 3) < 0)

### Exercise: The second-cheapest network

Write `second_mst_total(vertices, edges)`: the total of the cheapest spanning tree that is **different** from a minimum spanning tree (it may cost the same, if the MST isn't unique), or `None` if there isn't one. Find an MST with Kruskal's algorithm, then for each of its edges, find the MST with that edge left out; the answer is the smallest of those totals. Edges are `(u, v, weight)`.

In [ ]:
def second_mst_total(vertices, edges):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef all_trees(vs, es):\n    out = []\n    for combo in itertools.combinations(range(len(es)), len(vs) - 1):\n        parent = {v: v for v in vs}\n        def find(x):\n            while parent[x] != x:\n                x = parent[x]\n            return x\n        ok = True\n        for i in combo:\n            u, v, w = es[i]\n            if find(u) == find(v):\n                ok = False\n                break\n            parent[find(u)] = find(v)\n        if ok:\n            out.append(sum(es[i][2] for i in combo))\n    return sorted(out)\ndef test_second():\n    \"second best\"\n    edges = [(\"A\", \"B\", 1), (\"B\", \"C\", 2), (\"A\", \"C\", 3), (\"C\", \"D\", 4)]\n    assert second_mst_total(\"ABCD\", edges) == 8\n    assert second_mst_total(\"AB\", [(\"A\", \"B\", 5)]) is None\n    rng = random.Random(1)\n    for _ in range(60):\n        vs = list(range(rng.randint(2, 5)))\n        es = [(a, b, rng.randint(1, 9)) for a in vs for b in vs if a < b and rng.random() < 0.8]\n        trees = all_trees(vs, es)\n        if not trees:\n            continue\n        assert second_mst_total(vs, es) == (trees[1] if len(trees) > 1 else None)")

### Exercise: Escape with one wall broken

A grid of `.` and `#`; you may break through **at most one** wall. Write `escape_steps(grid)`: the fewest steps from the top-left to the bottom-right cell (4-way moves), or `None`. BFS over states `(cell, walls_broken)`.

In [ ]:
from collections import deque

def escape_steps(grid):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_escape():\n    \"one wall\"\n    assert escape_steps([\".#.\", \"###\", \"...\"]) == 4\n    assert escape_steps([\"..\", \"..\"]) == 2 and escape_steps([\".\"]) == 0\n    assert escape_steps([\".##\", \"###\", \"##.\"]) is None\n    assert escape_steps([\".#..\", \".#.#\", \"...#\", \"##..\"]) == 6")

### Exercise: Forge: a delivery planner

Write `RoadMap` to the specification above.

In [ ]:
import heapq

class RoadMap:
    def __init__(self, roads, one_way=()):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nROADS = [(\"Depot\", \"Mill\", 7), (\"Depot\", \"Square\", 4), (\"Square\", \"Mill\", 2), (\"Mill\", \"Farm\", 5),\n         (\"Square\", \"School\", 6), (\"School\", \"Farm\", 3), (\"Farm\", \"Lake\", 9), (\"School\", \"Lake\", 12)]\nONE_WAY = [(\"Lake\", \"Depot\", 8)]\ndef test_routes():\n    \"routes\"\n    m = RoadMap(ROADS, ONE_WAY)\n    assert m.places() == [\"Depot\", \"Farm\", \"Lake\", \"Mill\", \"School\", \"Square\"]\n    assert m.travel_time(\"Depot\", \"Farm\") == 11 and m.route(\"Depot\", \"Farm\") == [\"Depot\", \"Square\", \"Mill\", \"Farm\"]\n    assert m.travel_time(\"Lake\", \"Depot\") == 8 and m.travel_time(\"Depot\", \"Lake\") == 20\n    assert m.all_times(\"Square\") == {\"Square\": 0, \"Mill\": 2, \"Depot\": 4, \"School\": 6, \"Farm\": 7, \"Lake\": 16}\n    for bad in ([(\"A\", \"A\", 3)], [(\"A\", \"B\", 0)]):\n        try:\n            RoadMap(bad)\n            assert False\n        except ValueError:\n            pass\n    try:\n        m.travel_time(\"Depot\", \"Moon\")\n        assert False\n    except KeyError:\n        pass\ndef test_closures():\n    \"closing roads\"\n    m = RoadMap(ROADS, ONE_WAY)\n    m.close(\"Square\", \"Mill\")\n    assert m.travel_time(\"Depot\", \"Farm\") == 12 and m.route(\"Depot\", \"Farm\") == [\"Depot\", \"Mill\", \"Farm\"]\n    m.close(\"Depot\", \"Mill\")\n    assert m.travel_time(\"Depot\", \"Mill\") == 18\n    m.reopen(\"Square\", \"Mill\")\n    assert m.travel_time(\"Depot\", \"Mill\") == 6\n    m2 = RoadMap([(\"A\", \"B\", 1)])\n    m2.close(\"A\", \"B\")\n    assert m2.route(\"A\", \"B\") is None and m2.travel_time(\"A\", \"B\") is None\ndef test_depots():\n    \"depots and rounds\"\n    m = RoadMap(ROADS, ONE_WAY)\n    assert m.nearest_depot(\"Lake\", [\"Depot\", \"School\"]) == (\"School\", 12)\n    assert m.nearest_depot(\"Mill\", [\"Depot\", \"School\"]) == (\"Depot\", 6)\n    order, total = m.delivery_round(\"Depot\", [\"Farm\", \"School\", \"Mill\"])\n    assert order == [\"Mill\", \"Farm\", \"School\"] and total == 6 + 5 + 3 + 10\n    assert m.delivery_round(\"Depot\", []) == ([], 0)\ndef test_cables():\n    \"cables\"\n    m = RoadMap(ROADS, ONE_WAY)\n    total, used = m.cable_plan()\n    assert total == 2 + 3 + 4 + 5 + 9 and len(used) == 5\n    try:\n        RoadMap([(\"A\", \"B\", 1), (\"C\", \"D\", 1)]).cable_plan()\n        assert False\n    except ValueError:\n        pass\ndef test_random():\n    \"against Bellman-Ford\"\n    rng = random.Random(1)\n    for _ in range(40):\n        names = [f\"p{i}\" for i in range(7)]\n        roads = [(a, b, rng.randint(1, 9)) for i, a in enumerate(names) for b in names[i + 1:] if rng.random() < 0.3]\n        m = RoadMap(roads)\n        for s in m.places():\n            dist = {p: float(\"inf\") for p in m.places()}\n            dist[s] = 0\n            for _ in range(len(dist)):\n                for a, b, w in roads:\n                    dist[b] = min(dist[b], dist[a] + w)\n                    dist[a] = min(dist[a], dist[b] + w)\n            assert m.all_times(s) == {p: d for p, d in dist.items() if d < float(\"inf\")}")

> **Which algorithm, and why.** Every method in the Forge uses one of this week's algorithms, chosen by the shape of its question. Travel times have non-negative weights, so Dijkstra's algorithm. The cable plan wants everything connected cheaply, not short routes, so a minimum spanning tree, and only over two-way roads. The delivery round is a much harder question (visit everything, then return) that no efficient algorithm answers exactly, so the planner uses a reasonable greedy rule built from shortest paths, and says so. Recognising which kind of question you're facing, and when a problem is hard enough that a good approximation is the honest answer, matters as much as knowing the algorithms.

### Exercise: Forge bonus: an exchange desk

An exchange desk has rates for some pairs of currencies, and charges a fee as a fraction of every exchange. Write `ExchangeDesk(rates, fee=0.0)`, where `rates` maps `(from, to)` to the units of `to` per unit of `from`:

- Store each effective rate (rate × (1 − fee)) in `rates`, and every currency, sorted, in `currencies`. Raise `ValueError` for a fee outside 0 ≤ fee < 1, a rate that isn't positive, or a pair with the same currency twice.
- `loop()`: a list of currencies around which repeated exchanging **makes** money (the effective rates multiply to more than 1), or `None`. Bellman-Ford over weights −log(rate), starting every currency at 0, with a tolerance of 1e-12.
- `best(source, target)`: `(rate, path)`, the best overall rate from source to target (the largest product along a route) and its route, with Bellman-Ford; `None` if there's no route; `(1.0, [source])` when they're the same; `KeyError` for an unknown currency; and `ValueError` if a money-making loop exists (then "best" has no meaning).
- `convert(amount, path)`: the amount after exchanging along the path.

In [ ]:
import math

class ExchangeDesk:
    def __init__(self, rates, fee=0.0):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import math, itertools, random\nRATES = {(\"GBP\", \"USD\"): 1.25, (\"USD\", \"GBP\"): 0.79, (\"USD\", \"NGN\"): 1500.0, (\"NGN\", \"USD\"): 0.00066,\n         (\"GBP\", \"EUR\"): 1.16, (\"EUR\", \"GBP\"): 0.85, (\"EUR\", \"NGN\"): 1640.0, (\"NGN\", \"EUR\"): 0.0006}\ndef test_best():\n    \"best routes\"\n    desk = ExchangeDesk(RATES)\n    rate, path = desk.best(\"GBP\", \"NGN\")\n    assert path == [\"GBP\", \"EUR\", \"NGN\"] and math.isclose(rate, 1902.4)\n    assert math.isclose(desk.convert(100, path), 190240.0)\n    assert desk.best(\"GBP\", \"GBP\") == (1.0, [\"GBP\"])\n    assert ExchangeDesk({(\"A\", \"B\"): 2.0}).best(\"B\", \"A\") is None\n    for bad in ({(\"A\", \"A\"): 1.0}, {(\"A\", \"B\"): 0}):\n        try:\n            ExchangeDesk(bad)\n            assert False\n        except ValueError:\n            pass\ndef test_loops():\n    \"loops and fees\"\n    rigged = dict(RATES)\n    rigged[(\"NGN\", \"GBP\")] = 0.00054\n    desk = ExchangeDesk(rigged)\n    loop = desk.loop()\n    assert loop is not None and desk.convert(1.0, loop + [loop[0]]) > 1\n    try:\n        desk.best(\"GBP\", \"NGN\")\n        assert False\n    except ValueError:\n        pass\n    assert ExchangeDesk(rigged, fee=0.02).loop() is None, \"a 2% fee on every exchange kills the gain\"\n    assert ExchangeDesk(RATES).loop() is None\ndef test_random():\n    \"against trying every route\"\n    rng = random.Random(1)\n    for _ in range(40):\n        names = [\"A\", \"B\", \"C\", \"D\"]\n        values = {n: rng.uniform(0.5, 3) for n in names}\n        rates = {(a, b): values[b] / values[a] * rng.uniform(0.9, 1.0) for a in names for b in names if a != b and rng.random() < 0.7}\n        desk = ExchangeDesk(rates)\n        assert desk.loop() is None\n        best = None\n        for k in range(0, 3):\n            for mid in itertools.permutations(\"BC\", k):\n                p = (\"A\",) + mid + (\"D\",)\n                if all((x, y) in rates for x, y in zip(p, p[1:])):\n                    r = math.prod(rates[(x, y)] for x, y in zip(p, p[1:]))\n                    best = r if best is None else max(best, r)\n        got = desk.best(\"A\", \"D\") if \"A\" in desk.currencies and \"D\" in desk.currencies else None\n        if best is None:\n            assert got is None\n        else:\n            assert math.isclose(got[0], best, rel_tol=1e-9)")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

If any surprised you, revisit [Dijkstra's algorithm](w051d1.html), [Bellman-Ford](w051d2.html), [union-find](w051d3.html), [minimum spanning trees](w051d4.html) or [grids as graphs](w051d5.html).

## Mixed review

## Forge: a delivery planner

A district's roads take a number of minutes to drive; most are two-way, some one-way. A delivery firm wants a planner.

Write `RoadMap(roads, one_way=())`, where both are lists of `(a, b, minutes)`:

- Store the graph as `graph`, a dict of dicts (if two roads join the same pair, keep the quicker). Raise `ValueError` for a road from a place to itself or with minutes that aren't positive. Unknown places given to any method raise `KeyError`.
- `places()`: sorted.
- `travel_time(a, b)` and `route(a, b)`: the quickest time and route (`None` if unreachable), with Dijkstra's algorithm; explore neighbours in alphabetical order so routes are predictable.
- `all_times(start)`: a dict of the quickest time to every reachable place.
- `close(a, b)` and `reopen(a, b)`: close or reopen the road between a and b, **in both directions**; closed roads are ignored by every search until reopened.
- `nearest_depot(place, depots)`: `(depot, minutes)` for the depot that can reach `place` soonest (alphabetically first on a tie), or `None`.
- `delivery_round(depot, stops)`: a greedy round: from the depot, repeatedly drive to the **nearest** unvisited stop (by travel time, ties alphabetical), then back to the depot. Return `(order, total_minutes)`; raise `ValueError` if a stop or the depot can't be reached. (Finding the truly best round is the travelling salesperson problem, which no known algorithm solves quickly; greedy is a common, decent heuristic.)
- `cable_plan()`: fibre is laid along **two-way** roads only, and must connect every place. Return `(total_minutes_of_road_used, roads_used)` from Kruskal's algorithm, considering roads by length (ties by their names); raise `ValueError` if the two-way roads don't connect everything.

## Forge bonus: an exchange desk

## Weekly quiz

## Week 51 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can run Dijkstra's algorithm with a heap and lazy deletion, rebuild paths, and adapt it to widest paths and richer states. ([Day 1](w051d1.html))
- I can run Bellman-Ford, detect and recover negative cycles, limit paths to k edges, and solve difference constraints. ([Day 2](w051d2.html))
- I can keep weighted and parity union-find, detect contradictions, reverse deletions, and model percolation. ([Day 3](w051d3.html))
- I can find minimum spanning trees with Kruskal's and Prim's algorithms, and use them for clustering and bottlenecks. ([Day 4](w051d4.html))
- I can search grids with terrain costs, A*, bidirectional BFS and extra state. ([Day 5](w051d5.html))
- I built a delivery planner and an exchange desk. (Today)

Tomorrow is a **rest day**. Next week: **Greedy and backtracking**: greedy algorithms, interval problems, backtracking, N-queens and sudoku, and pruning.